In [755]:
import numpy as np
import pandas as pd

class topo2rest():
   def __init__(self, ifile:str, temps:list = [300.0, 500.0], nreps:int = 20, kappa:float = 1.00):
      '''Convert processed topology to REST2/3 input topology
         E_tot = gamma*E^{pp} + sqrt(gamma)*E^{pw} + E^{ww}
         gamma = T_0/T_i
         for REST2/3 bonds and angles are not scaled per the REST2 paper,
         however, for REST2 LJ parameters epsilon_i is scaled by epsilon_i*gamma
         for tempered atoms and all others are unmodified. 
         In the case of REST3, with the additon of sqrt(gamma)*kappa*E^{pw} which differs 
         from sqrt(gamma)*E^{pw}, we produced the combination rule 2 nonbonded terms
         involving protein-water interactions to override the pw nonbonded interactions
         as including gamma*kappa with each hot atom epsilon_i would result in the 
         incorrect form of: E_tot = gamma*kappa*E^{pp} + sqrt(gamma*kappa)*E^{pw} + E^{ww}:
         rather than the correct form: 
         E_tot = gamma*kappa*E^{pp} + sqrt(gamma)*kappa*E^{pw} + E^{ww}
         input
         ifile = inputtopology.top
         temps = ["lower temp":float, "upper temp":float ]; temperature range of replicas
         kappa:float = kappa scaling; if not equal to 1 REST3 implementation active
         hot_m = hot molecule; [0] for most systems will select the protein'''

      self.nreps = nreps
      self.kappa = kappa
      self.hot_m = None
      self.nmol = None
      self.molecules = None
      self.tempreps = self.compute_temperatures(temps)
      self.lambdai = self.compute_lambda()
      self.sections = {}
      self.sections_out = {}
      self.scaled_dihedrals = {}
      self.scaled_dihedral_types = {}
      self.scaled_atomtypes = {}
      self.hard_order_sections = [ "defaults", "atomtypes", "nonbond_params", "bondtypes", \
                                   "constrainttypes", "angletypes", "dihedraltypes", "molecules", "moleculetype"]
      with open(ifile) as topo:
         self.readfile = topo.readlines()
      self.gather_param_sections()
         
   def compute_lambda(self):
      return [ self.tempreps[0]/Ti for Ti in self.tempreps ]
   
   def compute_temperatures(self, temp_range:list):
      from numpy import log, exp
      tlow, thigh = temp_range
      temps = []
      for i in range(self.nreps):
         temps.append(tlow*exp((i)*log(thigh/tlow)/(self.nreps-1)))
      return temps
   
   def parse_section(self,trunks):
      first_round = True
      output = []
      for line in self.readfile[trunks:]:
         if "[" not in line and first_round!=True and len(line.split()) != 0:
            output.append(line)
         elif ";" == line[0]: 
            #output.append(line)
            continue
         elif "[" in line and first_round!=True: 
            break
         first_round = False
      return output

   def get_molecule_atomtypes(self,molecule:int=0):
      import pandas as pd
      from numpy import sqrt
      b=[]
      for i in self.sections['moleculetype'][molecule]['atoms']:
         if len(i.split())>1 and ';' not in i.split()[0] and '[' not in i :
            b.append(i.split()[:2])
      dataset = np.array(b,dtype=object)
      # Need to grab unique atoms
      atomtypes = dataset[:,1]
      return np.unique(atomtypes)
   
   def get_scale_nonbonded(self):
      from numpy import sqrt, float32
   
      def compute_se(e1:float,e2:float,s1:float,s2:float):
         eij = sqrt(float32(e1)*float32(e2))
         sij = 0.5*(float32(s1)+float32(s2))
         return eij,sij
      
      hot_atoms = np.array([i for j in self.hot_m for i in self.get_molecule_atomtypes(j) ])
      
      atomtypes_new = {}
      nonbonded_new = {}

      atomtypes = [i.split()[:-2] if i.split()[-2] == ';' else i.split()[:-1] if i.split()[-1] == ';' \
                               else i.split() for i in self.sections['atomtypes'] if ';' not in i[:3] if '[' not in i[:3] \
                               if '\n' not in i[:3]]
      #[print(len(at_),at_) for at_ in atomtypes]
      for lambdai in self.lambdai:
         at_out = []
         for _at in atomtypes:
            e_scaled = float32(lambdai) * float32(_at[-1])
            stringout = f'{_at[0]:<12} {_at[1]:<6} {_at[2]:>6} {_at[3]:>8}{_at[4]:^5}{_at[5]:>11}{_at[6]:>13}'
            at_out.append(stringout)
            stringout = f'{"s"+_at[0]:<12} {_at[1]:<6} {_at[2]:>6} {_at[3]:>8}{_at[4]:^5}{_at[5]:>11}{e_scaled:>13.5e}'
            at_out.append(stringout)
         atomtypes_new[lambdai] = at_out
         
         nbp_out = []
         for nbp in self.sections['nonbond_params']:
            nbp_ = nbp.split()
            if '[' in nbp_[0] or '\n' in nbp[:3]:
               continue
            elif ';' in nbp_[0]:
               #nbp_out.append(nbp)
               continue
            elif len(nbp.split()) == 5:
               nbp_out.append(nbp)
               check_hot = np.isin(np.array(nbp_[:2]), hot_atoms)
               if check_hot.all():
                  eps_scaled = float32(lambdai) * float32(nbp_[4])
                  stringout = f'{"s"+nbp_[0]:>5} {"s"+nbp_[1]:>4} {nbp_[2]:>5} {nbp_[3]:>10} {eps_scaled:>8.4f}'
                  nbp_out.append(stringout)
               elif check_hot.any():
                  if self.kappa == 1.0:
                     eps_scaled = lambdai * float32(nbp_[4])
                  elif check_hot.any():
                     eps_scaled = float32(self.kappa) * sqrt(float32(lambdai)) * float32(nbp_[4])
                  idx_ = np.where(check_hot)[0][0]
                  if idx_: stringout = f'{nbp_[0]:>5} {"s"+nbp_[1]:>4} {nbp_[2]:>5} {nbp_[3]:>10} {eps_scaled:>8.4f}' 
                  else: stringout = f'{"s"+nbp_[0]:>5} {nbp_[1]:>4} {nbp_[2]:>5} {nbp_[3]:>10} {eps_scaled:>8.4f}' 
                  nbp_out.append(stringout)
            else: print(f'here is the problem line: {nbp} {nbp_[0]}')
               
         if self.kappa != 1.00:
            print('kappa on')
            at_in = np.loadtxt(at_out,dtype=object)
            df_at = pd.DataFrame(at_in, columns=['name','atnum','mass','charge','ptype','sigma','epsilon'], dtype=object)
            print(df_at)
            for atom1 in df_at['name'].values:
               for atom2 in df_at['name'].values:
                  pos_atom1 = (df_at['name']==atom1)
                  pos_atom2 = (df_at['name']==atom2)
                  e1 = df_at[pos_atom1]['epsilon'].values[0]
                  e2 = df_at[pos_atom2]['epsilon'].values[0]
                  s1 = df_at[pos_atom1]['sigma'].values[0]
                  s2 = df_at[pos_atom2]['sigma'].values[0]
                  funct = str(1)
                  eij, sij = compute_se(e1,e2,s1,s2)
                  protein_solvent = np.isin(np.array([atom2[0], atom1[0]]), np.array(['s']))
                  if protein_solvent.any() and not protein_solvent.all():
                     scaled_eij = float32(self.kappa) * sqrt(float32(lambdai)) * eij
                     stringout = f'{"s"+atom1:>5} {atom2:>4} {funct:^5} {sij:>10.4f} {scaled_eij:>8.4f}'
                     nbp_out.append(stringout)
         nonbonded_new[lambdai] = nbp_out         
      self.sections_out['nonbond_params'] = nbp_out
      self.scaled_atomtypes = atomtypes_new
      pass
   
   def show_molecule_names(self):
      try:
         molecules = [" ".join(i.split()[:-1]) for i in self.sections['molecules'] if '[' not in i and ';' not in i.split()[0]] 
         molecules = {num: i for num,i in enumerate(molecules)}
         print("\n".join([f'{key}: {mol}' for key,mol in zip(molecules.keys(),molecules.values())]))
         if self.molecules == None: self.molecules = molecules
         if self.nmol == None: self.nmol = max(self.molecules.keys())
      except:
         print("Do you have molecules?")

   def get_molecule_atoms(self):
      import pandas as pd
      b=[]
      for i in self.sections['atoms']:
         i.split()
         if len(i.split())>1 and i.split()[0]!=';' and i.split()[0]!='[' :
            b.append(i.split())
   
   def _moleculetype_sub(self,linestart:int):
      first_round = True
      output = []
      for line in self.readfile[linestart:]:
         if "[" not in line and ';' not in line[:3]:
            output.append(line)
         elif ";" == line[0]:
            #output.append(line)
            continue
         elif "[" in line and first_round!=True: 
            break
         first_round = False
      return output 
   
   def identify_moltype_sections(self,trunks:int):
      section_start = []
      for i, line in enumerate(self.readfile[trunks:]):
         if '[' in line and 'moleculetype' not in line and 'system' not in line:
            section_start.append(i+trunks)
         elif i != 0 and 'moleculetype' in line or 'system' in line:
            break
      return section_start

   def parse_moleculetypes(self,trunks:int):
      first_round = True
      output = {}
      sections = self.identify_moltype_sections(trunks)
      output['header'] = self.readfile[trunks:trunks+2]
      for section in sections:
         section_ = self.readfile[section].split()[1]
         output[section_] = self._moleculetype_sub(section)
      return output
   
   def get_scale_dehedrals_(self, hot_m:list = [0]):

      dihedrals_new = {}
      dihedral_types_new = {}
      dihedral_types = [" ".join(i.split()[:-2]) if i.split()[-2] == ';' else i.split()[:-1] if i.split()[-1] == ';' \
                               else i.split() for i in self.sections['dihedraltypes'] if ';' not in i[:3] if '[' not in i[:3] \
                               if '\n' not in i[:3]]
      for hot in hot_m:
         dihedrals = self.sections['moleculetype'][hot]['dihedrals']
         for lambdai in self.lambdai:
            dih_new = []
            dih_types_new = []
            for dihedral in dihedrals:
               dls_ = dihedral.split()
               dls_ = [entry for entry in dls_ if len(entry) != 0]
               if len(dls_) == 5:
                  dih_new.append(dihedral)
               elif len(dls_) == 8:
                  Kscaled = float(dls_[6])*lambdai
                  stringout = f'{dls_[0]:>5} {dls_[1]:>5} {dls_[2]:>5} {dls_[3]:>5} {dls_[4]:^9}{dls_[6]:<10}{Kscaled:<10.3f}{dls_[7]}\n'
                  dih_new.append(stringout)
               else: 
                  print(f'Warning: incorrect parsing of dihedrals section\n expecting 5 or 8 columns found {len(dls_)}\n'+dihedral)
            dihedrals_new[lambdai] = dih_new
         self.scaled_dihedrals[hot] = dihedrals_new

      for lambdai in self.lambdai:
         for dihedraltype in dihedral_types:
            dtls_ = dihedraltype.split()
            Kscaled = float(dtls_[6])*lambdai
            if len(dtls_) == 8:
               dih_types_new.append(dihedraltype)
               check_atom_X = np.isin(np.array(dtls_[:4]), np.array(['X']))
               if sum(check_atom_X) == 0:
                  stringout = f'{"s"+dtls_[0]:>5} {"s"+dtls_[1]:>5} {"s"+dtls_[2]:>5} {"s"+dtls_[3]:>5} {dtls_[4]:^9}{dtls_[6]:<10}{Kscaled:<10.3f}{dtls_[7]}\n'
                  dih_types_new.append(stringout)
               elif dtls_[0] == 'X':
                  if sum(check_atom_X) == 1:
                     stringout = f'{dtls_[0]:>5} {"s"+dtls_[1]:>5} {"s"+dtls_[2]:>5} {"s"+dtls_[3]:>5} {dtls_[4]:^9}{dtls_[6]:<10}{Kscaled:<10.3f}{dtls_[7]}\n' 
                     dih_types_new.append(stringout)
                  elif dtls_[3] == 'X':
                     stringout = f'{dtls_[0]:>5} {"s"+dtls_[1]:>5} {"s"+dtls_[2]:>5} {dtls_[3]:>5} {dtls_[4]:^9}{dtls_[6]:<10}{Kscaled:<10.3f}{dtls_[7]}\n' 
                     dih_types_new.append(stringout)
                  elif dtls_[1] == 'X':
                     stringout = f'{dtls_[0]:>5} {dtls_[1]:>5} {"s"+dtls_[2]:>5} {"s"+dtls_[3]:>5} {dtls_[4]:^9}{dtls_[6]:<10}{Kscaled:<10.3f}{dtls_[7]}\n' 
                     dih_types_new.append(stringout)
                  else: print(f'In dihedraltype: something slipped through\n{dihedraltype}')
                  
               else: print(f'warning: parameter not found for {dtls_[:4]}')
            else: print('warning: dihedraltype not processed:\n '+dihedraltype)
         dihedral_types_new[lambdai] = dih_types_new
      self.scaled_dihedral_types = dihedral_types_new

   def gather_param_sections(self):
      for section in self.hard_order_sections[:-1]:
         is_select = [ i for i, line in enumerate(self.readfile) if section in line ]
         in_select = [f' [ {section} ] \n']
         for i in is_select:
            in_select += self.parse_section(i)
         self.sections[section]=in_select
      section = self.hard_order_sections[-1]
      is_select = [ i for i, line in enumerate(self.readfile) if section in line ] 
      moltype_dict = {} 
      for i in range(len(is_select)):
         moltype_dict[i] = self.parse_moleculetypes(is_select[i])
      self.sections[section] = moltype_dict
         

In [756]:
test = topo2rest('./example_topo/processed.top',kappa=1.06)

In [757]:
test.hot_m=[0]

In [758]:
test.sections['nonbond_params']

[' [ nonbond_params ] \n',
 '; i    j    funct    sigma   epsilon\n',
 '  OB   HB     1      0.150   1.2552\n']

In [759]:
test.get_scale_dehedrals_()

 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns found 0


 expecting 5 or 8 columns fo

In [760]:
test.hot_m,test.kappa

([0], 1.06)

In [761]:
test.get_scale_nonbonded()

kappa on
    name atnum      mass      charge ptype        sigma      epsilon
0     Br    35     79.90      0.0000     A  0.00000e+00  0.00000e+00
1    sBr    35     79.90      0.0000     A  0.00000e+00  0.00000e+00
2      C     6     12.01      0.0000     A  3.39967e-01  3.59824e-01
3     sC     6     12.01      0.0000     A  3.39967e-01  3.59824e-01
4     C6     6     12.01      0.0000     A  3.39967e-01  3.59824e-01
..   ...   ...       ...         ...   ...          ...          ...
163  sh1     1  1.008000  0.00000000     A   0.24219973  8.70272e-02
164   hc     1  1.008000  0.00000000     A    0.2600177    0.0870272
165  shc     1  1.008000  0.00000000     A    0.2600177  8.70272e-02
166   ho     1  1.008000  0.00000000     A  0.053792465    0.0196648
167  sho     1  1.008000  0.00000000     A  0.053792465  1.96648e-02

[168 rows x 7 columns]
kappa on
    name atnum      mass      charge ptype        sigma      epsilon
0     Br    35     79.90      0.0000     A  0.00000e+00  0.00

In [762]:
test.sections_out['nonbond_params']

['  OB   HB     1      0.150   1.2552\n',
 '  sOB  sHB     1      0.150   0.7531',
 '  sBr  sBr   1       0.0000   0.0000',
 '  sBr   sC   1       0.1700   0.0000',
 '  sBr  sC6   1       0.1700   0.0000',
 '  sBr  sC5   1       0.1700   0.0000',
 '  sBr  sCA   1       0.1700   0.0000',
 '  sBr  sCB   1       0.1700   0.0000',
 '  sBr  sCC   1       0.1700   0.0000',
 '  sBr  sCK   1       0.1700   0.0000',
 '  sBr  sCM   1       0.1700   0.0000',
 '  sBr  sCN   1       0.1700   0.0000',
 '  sBr  sCQ   1       0.1700   0.0000',
 '  sBr  sCR   1       0.1700   0.0000',
 '  sBr  sCT   1       0.1700   0.0000',
 '  sBr  sC1   1       0.1700   0.0000',
 '  sBr  sC3   1       0.1700   0.0000',
 '  sBr  sC4   1       0.1700   0.0000',
 '  sBr  sC7   1       0.1700   0.0000',
 '  sBr  sC8   1       0.1700   0.0000',
 '  sBr  sC9   1       0.1700   0.0000',
 '  sBr  sCV   1       0.1700   0.0000',
 '  sBr  sCW   1       0.1700   0.0000',
 '  sBr  sC*   1       0.1700   0.0000',
 '  sBr  sC0   

In [643]:
test.show_molecule_names()

0: Protein_chain_A
1: SOL
2: NA
3: CL


In [646]:
test.scaled_dihedral_types

{0: {1.0: ['C C1 N O 4 180.0 4.60240 2',
   '   sC   sC1    sN    sO     4    4.60240   4.602     2\n',
   'C C1 N OB 4 180.0 4.60240 2',
   '   sC   sC1    sN   sOB     4    4.60240   4.602     2\n',
   'C C1 N H 4 180.0 4.60240 2',
   '   sC   sC1    sN    sH     4    4.60240   4.602     2\n',
   'C C1 N HB 4 180.0 4.60240 2',
   '   sC   sC1    sN   sHB     4    4.60240   4.602     2\n',
   'C9 O C OH 4 180.0 43.93200 2',
   '  sC9    sO    sC   sOH     4    43.93200  43.932    2\n',
   'CB CK N* CT 4 180.0 4.18400 2',
   '  sCB   sCK   sN*   sCT     4    4.18400   4.184     2\n',
   'C CM N* CT 4 180.0 4.18400 2',
   '   sC   sCM   sN*   sCT     4    4.18400   4.184     2\n',
   'CT O C OH 4 180.0 43.93200 2',
   '  sCT    sO    sC   sOH     4    43.93200  43.932    2\n',
   'CT CV CC NA 4 180.0 4.60240 2',
   '  sCT   sCV   sCC   sNA     4    4.60240   4.602     2\n',
   'CT CW CC NB 4 180.0 4.60240 2',
   '  sCT   sCW   sCC   sNB     4    4.60240   4.602     2\n',
   'CT CC CW NB

In [649]:
test.scaled_dihedrals

dict_keys([0])